# Multi-Leg Vessel-Cargo Optimization (Cargill Hackathon)

This notebook implements a **multi-leg routing optimization** that:
- **Guarantees all 3 committed cargoes are lifted** (exactly once)
- Allows **Cargill vessels to execute multiple cargoes sequentially** (including optional market cargoes)
- Uses **market vessels only for committed cargoes**
- Computes **leg-by-leg fuel burn with time-varying bunker prices** (forward curve)
- **Explicitly recommends which market cargoes to take** and where in each vessel's sequence

## Key Features
- **Sequential legs**: Vessels can carry one cargo at a time, but chain multiple cargoes
- **Waiting allowed**: Vessels can wait at anchorage if arriving before laycan start
- **Realistic distances**: Uses searoute-calculated maritime distances
- **Time-varying fuel costs**: Bunker prices from forward curve based on actual voyage dates

In [14]:
# Cell 1: Install required libraries
# Run this cell first - it will install dependencies if not already installed
%pip install ortools pandas folium matplotlib -q

print("✓ Libraries installed/verified")

Note: you may need to restart the kernel to use updated packages.
✓ Libraries installed/verified


In [2]:
# Cell 2: Load all CSVs
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import re

base = Path("data")

# Load vessel data
print("Loading vessel data...")
cargill_vessels = pd.read_csv(base / "Cargill_Capesize_Vessels.csv")
market_vessels = pd.read_csv(base / "Market_Vessels_Formatted.csv")

# Load cargo data
print("Loading cargo data...")
cargill_cargoes = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")

# Load port distances
print("Loading port distances...")
port_distances = pd.read_csv(base / "Port Distances.csv")

# Load port locations
print("Loading port locations...")
port_locations = pd.read_csv(base / "port_locations.csv")

# Load bunker forward curve
print("Loading bunker forward curve...")
bunker_forward_curve = pd.read_csv(base / "bunker_forward_curve.csv")

# Load freight rates
print("Loading freight rates...")
freight_rates = pd.read_csv(base / "freight_rates.csv")

print("\n✓ All data loaded successfully!")
print(f"Cargill vessels: {len(cargill_vessels)}")
print(f"Market vessels: {len(market_vessels)}")
print(f"Cargill cargoes: {len(cargill_cargoes)}")
print(f"Market cargoes: {len(market_cargoes)}")
print(f"Port distances: {len(port_distances)}")
print(f"Bunker locations: {len(bunker_forward_curve)}")
print(f"Freight routes: {len(freight_rates)}")

Loading vessel data...
Loading cargo data...
Loading port distances...
Loading port locations...
Loading bunker forward curve...
Loading freight rates...

✓ All data loaded successfully!
Cargill vessels: 4
Market vessels: 11
Cargill cargoes: 3
Market cargoes: 8
Port distances: 15661
Bunker locations: 18
Freight routes: 4


In [3]:
# Cell 3: Preprocess data (NO PORT NORMALIZATION - using raw CSV values)

# Preprocess vessels
print("Preprocessing vessels...")

# Cargill vessels
cargill_vessels_processed = cargill_vessels.copy()
cargill_vessels_processed['vessel_name'] = cargill_vessels_processed['Vessel Name']
cargill_vessels_processed['deadweight_tonnage_dwt'] = cargill_vessels_processed['DWT (MT)']
cargill_vessels_processed['hire_rate_usd_per_day'] = cargill_vessels_processed['Hire Rate (USD/day)']
cargill_vessels_processed['economical_speed_knots'] = cargill_vessels_processed['Economical Speed Ballast (kn)']
cargill_vessels_processed['sea_consumption_mt_per_day'] = cargill_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
cargill_vessels_processed['port_consumption_mt_per_day'] = cargill_vessels_processed['Port Consumption Working (mt/day)']
cargill_vessels_processed['port_consumption_idle_mt_per_day'] = cargill_vessels_processed['Port Consumption Idle (mt/day)']
cargill_vessels_processed['current_position_port'] = cargill_vessels_processed['Current Position / Status']
cargill_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(cargill_vessels_processed['ETD'])
cargill_vessels_processed['speed_laden'] = cargill_vessels_processed['Economical Speed Laden (kn)']
cargill_vessels_processed['consumption_laden'] = cargill_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Market vessels
market_vessels_processed = market_vessels.copy()
market_vessels_processed['vessel_name'] = market_vessels_processed['Vessel Name']
market_vessels_processed['deadweight_tonnage_dwt'] = market_vessels_processed['DWT (MT)']
market_vessels_processed['hire_rate_usd_per_day'] = 0  # Market vessels - will use freight rate instead
market_vessels_processed['economical_speed_knots'] = market_vessels_processed['Economical Speed Ballast (kn)']
market_vessels_processed['sea_consumption_mt_per_day'] = market_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
market_vessels_processed['port_consumption_mt_per_day'] = market_vessels_processed['Port Consumption Working (mt/day)']
market_vessels_processed['port_consumption_idle_mt_per_day'] = market_vessels_processed['Port Consumption Idle (mt/day)']
market_vessels_processed['current_position_port'] = market_vessels_processed['Current Position / Status']
market_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(market_vessels_processed['ETD'])
market_vessels_processed['speed_laden'] = market_vessels_processed['Economical Speed Laden (kn)']
market_vessels_processed['consumption_laden'] = market_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Preprocess cargoes
print("Preprocessing cargoes...")

# Cargill cargoes
cargill_cargoes_processed = cargill_cargoes.copy()
cargill_cargoes_processed['cargo_id'] = ['CARGILL_' + str(i+1) for i in range(len(cargill_cargoes_processed))]
cargill_cargoes_processed['quantity_mt'] = cargill_cargoes_processed['Quantity_MT']
cargill_cargoes_processed['freight_rate_usd_per_mt'] = cargill_cargoes_processed['Freight_Rate_USD_PMT']
cargill_cargoes_processed['commission_percent'] = cargill_cargoes_processed['Commission_Percent'] / 100.0
cargill_cargoes_processed['laycan_start_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_Start'])
cargill_cargoes_processed['laycan_end_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_End'])
cargill_cargoes_processed['load_port'] = cargill_cargoes_processed['Load_Port_Primary']
cargill_cargoes_processed['discharge_port'] = cargill_cargoes_processed['Discharge_Port_Primary']
cargill_cargoes_processed['port_costs_usd'] = cargill_cargoes_processed['Port_Cost_USD'].fillna(0)
cargill_cargoes_processed['load_turn_time_hours'] = cargill_cargoes_processed['Load_Turn_Time_Hours'].fillna(12)
cargill_cargoes_processed['discharge_turn_time_hours'] = cargill_cargoes_processed['Discharge_Turn_Time_Hours'].fillna(24)
cargill_cargoes_processed['route'] = cargill_cargoes_processed['Route']

# Market cargoes
market_cargoes_processed = market_cargoes.copy()
market_cargoes_processed['cargo_id'] = ['MARKET_' + str(i+1) for i in range(len(market_cargoes_processed))]
market_cargoes_processed['quantity_mt'] = market_cargoes_processed['Quantity_MT']
market_cargoes_processed['freight_rate_usd_per_mt'] = 0  # Will be filled from freight_rates.csv
market_cargoes_processed['commission_percent'] = market_cargoes_processed['Commission_Percent'] / 100.0
market_cargoes_processed['laycan_start_date'] = pd.to_datetime(market_cargoes_processed['Laycan_Start'])
market_cargoes_processed['laycan_end_date'] = pd.to_datetime(market_cargoes_processed['Laycan_End'])
market_cargoes_processed['load_port'] = market_cargoes_processed['Load_Port']
market_cargoes_processed['discharge_port'] = market_cargoes_processed['Discharge_Port']
market_cargoes_processed['port_costs_usd'] = (
    market_cargoes_processed['Port_Cost_Load_USD'].fillna(0) + 
    market_cargoes_processed['Port_Cost_Discharge_USD'].fillna(0)
)
market_cargoes_processed['load_turn_time_hours'] = market_cargoes_processed['Load_Turn_Time_hr'].fillna(12)
market_cargoes_processed['discharge_turn_time_hours'] = market_cargoes_processed['Discharge_Turn_Time_hr'].fillna(24)
market_cargoes_processed['route'] = market_cargoes_processed['Route']

# Build distance lookup from Port Distances.csv using raw port names
print("Building distance lookup from Port Distances.csv...")
port_distances_processed = port_distances.copy()
port_distances_processed['port_from'] = port_distances_processed['PORT_NAME_FROM'].str.strip()
port_distances_processed['port_to'] = port_distances_processed['PORT_NAME_TO'].str.strip()
port_distances_processed['distance_nautical_miles'] = port_distances_processed['DISTANCE']

distance_lookup = {}
for _, row in port_distances_processed.iterrows():
    port_from = row['port_from']
    port_to = row['port_to']
    distance = row['distance_nautical_miles']
    key = (port_from, port_to)
    distance_lookup[key] = distance
    key_reverse = (port_to, port_from)
    if key_reverse not in distance_lookup:
        distance_lookup[key_reverse] = distance

print("\n✓ Data preprocessing complete!")
print(f"Processed {len(cargill_vessels_processed)} Cargill vessels")
print(f"Processed {len(market_vessels_processed)} market vessels")
print(f"Processed {len(cargill_cargoes_processed)} Cargill cargoes")
print(f"Processed {len(market_cargoes_processed)} market cargoes")
print(f"Distance lookup entries: {len(distance_lookup)}")

Preprocessing vessels...
Preprocessing cargoes...
Building distance lookup from Port Distances.csv...

✓ Data preprocessing complete!
Processed 4 Cargill vessels
Processed 11 market vessels
Processed 3 Cargill cargoes
Processed 8 market cargoes
Distance lookup entries: 30896


In [4]:
# Cell 4: Build bunker price lookup function (same as original notebook)

# Reshape bunker forward curve from wide to long format
bunker_long = []
date_cols = [col for col in bunker_forward_curve.columns if re.match(r'^\d{4}-\d{2}-\d{2}$', col)]

for _, row in bunker_forward_curve.iterrows():
    location = row['location']
    fuel_grade = row['fuel_grade']
    lat = row['latitude']
    lon = row['longitude']
    for date_col in date_cols:
        price = row[date_col]
        if pd.notna(price):
            date = pd.to_datetime(date_col)
            bunker_long.append({
                'location': location,
                'fuel_grade': fuel_grade,
                'latitude': lat,
                'longitude': lon,
                'date': date,
                'price': price
            })

bunker_df = pd.DataFrame(bunker_long)
bunker_df = bunker_df.sort_values(['location', 'fuel_grade', 'date'])

# Create location mapping from port names to bunker locations
port_to_bunker_location = {}
for _, port_row in port_locations.iterrows():
    port_name = str(port_row['port_name']).strip()
    port_lat = port_row['latitude']
    port_lon = port_row['longitude']
    
    min_dist = float('inf')
    nearest_location = None
    for _, bunker_row in bunker_forward_curve.iterrows():
        bunker_lat = bunker_row['latitude']
        bunker_lon = bunker_row['longitude']
        dist = ((port_lat - bunker_lat)**2 + (port_lon - bunker_lon)**2)**0.5
        if dist < min_dist:
            min_dist = dist
            nearest_location = bunker_row['location']
    
    port_to_bunker_location[port_name] = nearest_location

def get_bunker_price(port_name, fuel_grade='VLSFO', date=None):
    """Get bunker price for a port and date."""
    if date is None:
        return None
    
    if isinstance(date, datetime):
        date = date.date()
    elif isinstance(date, pd.Timestamp):
        date = date.date()
    
    port_clean = str(port_name).strip() if port_name else None
    if not port_clean:
        return None
    
    bunker_location = None
    if port_clean in port_to_bunker_location:
        bunker_location = port_to_bunker_location[port_clean]
    
    if not bunker_location:
        matching = bunker_df[(bunker_df['fuel_grade'] == fuel_grade) & (bunker_df['date'] == pd.Timestamp(date))]
        if len(matching) > 0:
            return matching['price'].mean()
        return None
    
    location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                 (bunker_df['fuel_grade'] == fuel_grade)].copy()
    
    if len(location_prices) == 0:
        if fuel_grade == 'MGO':
            location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                         (bunker_df['fuel_grade'] == 'VLSFO')].copy()
        if len(location_prices) == 0:
            return None
    
    location_prices = location_prices.sort_values('date')
    target_date = pd.Timestamp(date)
    
    exact = location_prices[location_prices['date'] == target_date]
    if len(exact) > 0:
        return exact.iloc[0]['price']
    
    if target_date < location_prices['date'].min():
        return location_prices.iloc[0]['price']
    
    if target_date > location_prices['date'].max():
        return location_prices.iloc[-1]['price']
    
    before = location_prices[location_prices['date'] < target_date].iloc[-1]
    after = location_prices[location_prices['date'] > target_date].iloc[0]
    
    days_between = (after['date'] - before['date']).days
    days_to_target = (target_date - before['date']).days
    
    if days_between == 0:
        return before['price']
    
    weight = days_to_target / days_between
    interpolated_price = before['price'] * (1 - weight) + after['price'] * weight
    
    return interpolated_price

print("✓ Bunker price lookup function ready")

✓ Bunker price lookup function ready


In [5]:
# Cell 5: Build freight rate lookup function (same as original notebook)

# Reshape freight rates from wide to long format
freight_long = []
period_cols = [col for col in freight_rates.columns if col != 'route']

for _, row in freight_rates.iterrows():
    route = row['route']
    for period_col in period_cols:
        rate = row[period_col]
        if pd.notna(rate) and rate != '':
            freight_long.append({
                'route': route,
                'period': period_col,
                'rate': rate
            })

freight_rates_long = pd.DataFrame(freight_long)

def match_route_to_freight_rate(cargo_route):
    """Match a cargo route to a freight rate route."""
    if pd.isna(cargo_route):
        return None
    
    route_str = str(cargo_route).strip()
    
    route_mapping = {
        'Brazil – China': 'C3 (Tubarao-Qingdao)',
        'Brazil – China (Iron Ore)': 'C3 (Tubarao-Qingdao)',
        'Australia – China': 'C5 (West Australia-Qingdao)',
        'Australia – China (Iron Ore)': 'C5 (West Australia-Qingdao)',
        'West Africa – China': None,
        'West Africa – India': None,
        'South Africa – China': None,
        'Indonesia – India': None,
        'Canada – China': None,
        'Australia – South Korea': None,
        'Brazil – Malaysia': None,
    }
    
    if route_str in route_mapping:
        return route_mapping[route_str]
    
    route_upper = route_str.upper()
    for freight_route in freight_rates['route'].unique():
        freight_route_clean = str(freight_route).strip().upper()
        if 'BRAZIL' in route_upper and 'CHINA' in route_upper and 'C3' in freight_route_clean:
            return freight_route
        if 'AUSTRALIA' in route_upper and 'CHINA' in route_upper and 'C5' in freight_route_clean:
            return freight_route
    
    return None

def get_market_freight_rate(cargo_route, laycan_date):
    """Get market freight rate for a cargo route and laycan period."""
    matched_route = match_route_to_freight_rate(cargo_route)
    if not matched_route:
        return None
    
    if isinstance(laycan_date, datetime):
        laycan_date = laycan_date.date()
    elif isinstance(laycan_date, pd.Timestamp):
        laycan_date = laycan_date.date()
    
    year = laycan_date.year
    month = laycan_date.month
    quarter = (month - 1) // 3 + 1
    
    period_candidates = [
        f"{year}-{month:02d}",
        f"{year}-Q{quarter}",
        str(year),
    ]
    
    for period in period_candidates:
        matching = freight_rates_long[
            (freight_rates_long['route'] == matched_route) & 
            (freight_rates_long['period'] == period)
        ]
        if len(matching) > 0:
            return matching.iloc[0]['rate']
    
    route_rates = freight_rates_long[freight_rates_long['route'] == matched_route]
    if len(route_rates) > 0:
        return route_rates['rate'].iloc[0]
    
    return None

# Fill market cargo freight rates
for idx, cargo in market_cargoes_processed.iterrows():
    market_rate = get_market_freight_rate(cargo['route'], cargo['laycan_start_date'])
    if market_rate is not None:
        market_cargoes_processed.at[idx, 'freight_rate_usd_per_mt'] = market_rate

print("✓ Freight rate lookup function ready")

✓ Freight rate lookup function ready


In [6]:
# Cell 6: Implement generalized leg evaluator (with waiting logic)

def get_distance(port_from, port_to, distance_lookup):
    """Get distance between two ports using raw port names."""
    if not port_from or not port_to:
        return None
    
    port_from_clean = str(port_from).strip()
    port_to_clean = str(port_to).strip()
    
    distance = distance_lookup.get((port_from_clean, port_to_clean))
    if distance is not None:
        return distance
    
    distance = distance_lookup.get((port_to_clean, port_from_clean))
    if distance is not None:
        return distance
    
    return None

def evaluate_leg(start_port, start_time, vessel_row, cargo_row, distance_lookup, get_bunker_price_fn, get_market_freight_rate_fn):
    """
    Evaluate a single leg: vessel goes from start_port at start_time to execute cargo.
    
    Args:
        start_port: Port name where vessel starts (raw string)
        start_time: datetime when vessel departs start_port
        vessel_row: Vessel dataframe row
        cargo_row: Cargo dataframe row
        distance_lookup: Dictionary of port-to-port distances
        get_bunker_price_fn: Function to get bunker price
        get_market_freight_rate_fn: Function to get market freight rate
    
    Returns:
        dict with leg details and profit, or None if infeasible
    """
    try:
        # 1. Capacity check
        if cargo_row['quantity_mt'] > vessel_row['deadweight_tonnage_dwt']:
            return None
        
        # 2. Distance lookups
        ballast_port_from = start_port
        ballast_port_to = cargo_row['load_port']
        laden_port_from = cargo_row['load_port']
        laden_port_to = cargo_row['discharge_port']
        
        if not ballast_port_from or not ballast_port_to or not laden_port_to:
            return None
        
        ballast_distance = get_distance(ballast_port_from, ballast_port_to, distance_lookup)
        laden_distance = get_distance(laden_port_from, laden_port_to, distance_lookup)
        
        if ballast_distance is None or laden_distance is None:
            return None
        
        # 3. Time calculations
        # Ballast leg
        ballast_speed = vessel_row['economical_speed_knots']
        days_ballast = ballast_distance / (ballast_speed * 24.0)
        
        # Arrival at load port
        if isinstance(start_time, str):
            start_time = pd.to_datetime(start_time)
        elif isinstance(start_time, pd.Timestamp):
            start_time = start_time.to_pydatetime()
        
        arrival_at_load = start_time + timedelta(days=days_ballast)
        
        # 4. Waiting logic (if arriving before laycan start)
        laycan_start = cargo_row['laycan_start_date']
        if isinstance(laycan_start, pd.Timestamp):
            laycan_start = laycan_start.to_pydatetime()
        
        laycan_end = cargo_row['laycan_end_date']
        if isinstance(laycan_end, pd.Timestamp):
            laycan_end = laycan_end.to_pydatetime()
        
        waiting_days = 0.0
        actual_load_start = arrival_at_load
        
        if arrival_at_load.date() < laycan_start.date():
            # Arrive early - wait until laycan start
            waiting_days = (laycan_start.date() - arrival_at_load.date()).days
            actual_load_start = laycan_start
        elif arrival_at_load.date() > laycan_end.date():
            # Arrive too late - infeasible
            return None
        
        # Laden leg
        laden_speed = vessel_row['speed_laden']
        days_laden = laden_distance / (laden_speed * 24.0)
        
        # Port time
        port_days = (cargo_row['load_turn_time_hours'] + cargo_row['discharge_turn_time_hours']) / 24.0
        
        # Total time
        total_days = days_ballast + waiting_days + port_days + days_laden
        
        # Completion time
        completion_time = actual_load_start + timedelta(days=port_days + days_laden)
        
        # 5. Revenue
        freight_rate = cargo_row['freight_rate_usd_per_mt']
        quantity = cargo_row['quantity_mt']
        revenue = freight_rate * quantity
        
        # 6. Costs
        # Commission
        commission = cargo_row['commission_percent'] * revenue
        
        # Fuel costs (time-varying based on actual dates)
        # Ballast fuel (priced at start_port, start_time)
        fuel_ballast = vessel_row['sea_consumption_mt_per_day'] * days_ballast
        bunker_price_ballast = get_bunker_price_fn(ballast_port_from, 'VLSFO', start_time.date())
        if bunker_price_ballast is None:
            bunker_price_ballast = 500  # Default fallback
        
        # Waiting fuel (idle consumption at load port, priced at load port, arrival date)
        fuel_waiting = vessel_row['port_consumption_idle_mt_per_day'] * waiting_days
        bunker_price_waiting = get_bunker_price_fn(ballast_port_to, 'VLSFO', arrival_at_load.date())
        if bunker_price_waiting is None:
            bunker_price_waiting = 500
        
        # Laden fuel (priced at load port, departure date)
        fuel_laden = vessel_row['consumption_laden'] * days_laden
        bunker_price_laden = get_bunker_price_fn(laden_port_from, 'VLSFO', actual_load_start.date())
        if bunker_price_laden is None:
            bunker_price_laden = 500
        
        # Port fuel (working consumption during load/discharge)
        fuel_port = vessel_row['port_consumption_mt_per_day'] * port_days
        bunker_price_port = get_bunker_price_fn(laden_port_from, 'VLSFO', actual_load_start.date())
        if bunker_price_port is None:
            bunker_price_port = 500
        
        fuel_cost = (fuel_ballast * bunker_price_ballast + 
                    fuel_waiting * bunker_price_waiting +
                    fuel_laden * bunker_price_laden + 
                    fuel_port * bunker_price_port)
        
        # Hire cost
        hire_cost = vessel_row['hire_rate_usd_per_day'] * total_days
        
        # Port costs
        port_costs = cargo_row['port_costs_usd']
        
        total_costs = commission + fuel_cost + hire_cost + port_costs
        
        # Profit
        profit = revenue - total_costs
        
        # Opportunity cost (market rate comparison)
        market_rate = get_market_freight_rate_fn(cargo_row.get('route'), cargo_row['laycan_start_date'])
        opportunity_cost = None
        if market_rate is not None:
            market_revenue = market_rate * quantity
            opportunity_cost = market_revenue - revenue
        
        return {
            'profit': profit,
            'revenue': revenue,
            'costs': total_costs,
            'commission': commission,
            'fuel_cost': fuel_cost,
            'fuel_cost_ballast': fuel_ballast * bunker_price_ballast,
            'fuel_cost_waiting': fuel_waiting * bunker_price_waiting,
            'fuel_cost_laden': fuel_laden * bunker_price_laden,
            'fuel_cost_port': fuel_port * bunker_price_port,
            'hire_cost': hire_cost,
            'port_costs': port_costs,
            'opportunity_cost': opportunity_cost,
            'market_rate': market_rate,
            'offered_rate': freight_rate,
            'days_ballast': days_ballast,
            'days_waiting': waiting_days,
            'days_laden': days_laden,
            'days_port': port_days,
            'total_days': total_days,
            'ballast_distance': ballast_distance,
            'laden_distance': laden_distance,
            'arrival_at_load': arrival_at_load,
            'actual_load_start': actual_load_start,
            'completion_time': completion_time,
            'end_port': laden_port_to,
            'end_time': completion_time,
            'bunker_price_ballast': bunker_price_ballast,
            'bunker_price_waiting': bunker_price_waiting,
            'bunker_price_laden': bunker_price_laden,
            'bunker_price_port': bunker_price_port
        }
    
    except Exception as e:
        print(f"Error evaluating leg: {e}")
        import traceback
        traceback.print_exc()
        return None

print("✓ Leg evaluator function implemented")

✓ Leg evaluator function implemented


In [7]:
# Cell 7: Generate feasible arcs for multi-leg routing

print("Generating feasible arcs for multi-leg routing...")
print("=" * 80)

# Combine all cargoes for easier indexing
all_cargoes = pd.concat([
    cargill_cargoes_processed.assign(cargo_type='committed', cargo_idx_in_type=range(len(cargill_cargoes_processed))),
    market_cargoes_processed.assign(cargo_type='market', cargo_idx_in_type=range(len(market_cargoes_processed)))
], ignore_index=True).reset_index(drop=True)

# Create cargo index mapping
cargo_to_idx = {}
for idx, cargo in all_cargoes.iterrows():
    cargo_id = cargo['cargo_id']
    cargo_to_idx[cargo_id] = idx

print(f"Total cargoes: {len(all_cargoes)} ({len(cargill_cargoes_processed)} committed, {len(market_cargoes_processed)} market)")

# Generate arcs for Cargill vessels
cargill_arcs = []

for v_idx, vessel in cargill_vessels_processed.iterrows():
    vessel_name = vessel['vessel_name']
    start_port = vessel['current_position_port']
    start_time = vessel['estimated_time_of_departure']
    
    print(f"\nVessel: {vessel_name}")
    print(f"  Start: {start_port} at {start_time.date()}")
    
    # Arcs from Start -> any cargo
    for c_idx, cargo in all_cargoes.iterrows():
        leg_data = evaluate_leg(
            start_port, start_time, vessel, cargo,
            distance_lookup, get_bunker_price, get_market_freight_rate
        )
        
        if leg_data is not None:
            cargill_arcs.append({
                'vessel_idx': v_idx,
                'vessel_name': vessel_name,
                'from_node': 'START',
                'from_port': start_port,
                'from_time': start_time,
                'to_node': cargo['cargo_id'],
                'to_cargo_idx': c_idx,
                'cargo_type': cargo['cargo_type'],
                'leg_data': leg_data,
                'profit': leg_data['profit'],
                'end_port': leg_data['end_port'],
                'end_time': leg_data['end_time']
            })
    
    # Arcs from cargo -> cargo (for chaining)
    # For each cargo that this vessel can reach, check if it can chain to another cargo
    for c1_idx, cargo1 in all_cargoes.iterrows():
        # Check if vessel can reach cargo1 from start
        leg1_data = evaluate_leg(
            start_port, start_time, vessel, cargo1,
            distance_lookup, get_bunker_price, get_market_freight_rate
        )
        
        if leg1_data is not None:
            # Now check if we can chain cargo2 after cargo1
            end_port_c1 = leg1_data['end_port']
            end_time_c1 = leg1_data['end_time']
            
            for c2_idx, cargo2 in all_cargoes.iterrows():
                if c1_idx == c2_idx:
                    continue  # Can't chain to itself
                
                leg2_data = evaluate_leg(
                    end_port_c1, end_time_c1, vessel, cargo2,
                    distance_lookup, get_bunker_price, get_market_freight_rate
                )
                
                if leg2_data is not None:
                    cargill_arcs.append({
                        'vessel_idx': v_idx,
                        'vessel_name': vessel_name,
                        'from_node': cargo1['cargo_id'],
                        'from_port': end_port_c1,
                        'from_time': end_time_c1,
                        'to_node': cargo2['cargo_id'],
                        'to_cargo_idx': c2_idx,
                        'cargo_type': cargo2['cargo_type'],
                        'leg_data': leg2_data,
                        'profit': leg2_data['profit'],
                        'end_port': leg2_data['end_port'],
                        'end_time': leg2_data['end_time']
                    })

print(f"\n✓ Generated {len(cargill_arcs)} feasible arcs for Cargill vessels")

# Generate arcs for Market vessels (only Start -> committed cargoes)
market_arcs = []

for m_idx, vessel in market_vessels_processed.iterrows():
    vessel_name = vessel['vessel_name']
    start_port = vessel['current_position_port']
    start_time = vessel['estimated_time_of_departure']
    
    # Only committed cargoes for market vessels
    for c_idx, cargo in cargill_cargoes_processed.iterrows():
        leg_data = evaluate_leg(
            start_port, start_time, vessel, cargo,
            distance_lookup, get_bunker_price, get_market_freight_rate
        )
        
        if leg_data is not None:
            market_arcs.append({
                'vessel_idx': m_idx,
                'vessel_name': vessel_name,
                'from_node': 'START',
                'from_port': start_port,
                'from_time': start_time,
                'to_node': cargo['cargo_id'],
                'to_cargo_idx': c_idx,
                'cargo_type': 'committed',
                'leg_data': leg_data,
                'profit': leg_data['profit'],
                'end_port': leg_data['end_port'],
                'end_time': leg_data['end_time']
            })

print(f"✓ Generated {len(market_arcs)} feasible arcs for Market vessels")
print(f"\nTotal feasible arcs: {len(cargill_arcs) + len(market_arcs)}")

Generating feasible arcs for multi-leg routing...
Total cargoes: 11 (3 committed, 8 market)

Vessel: ANN BELL
  Start: QINGDAO at 2026-02-25

Vessel: OCEAN HORIZON
  Start: MAP TA PHUT at 2026-03-01

Vessel: PACIFIC GLORY
  Start: GWANGYANG LNG TERMINAL at 2026-03-10

Vessel: GOLDEN ASCENT
  Start: FANGCHENG at 2026-03-08

✓ Generated 20 feasible arcs for Cargill vessels
✓ Generated 10 feasible arcs for Market vessels

Total feasible arcs: 30


In [8]:
# Cell 8: Implement CP-SAT flow model for multi-leg optimization

from ortools.sat.python import cp_model

print("Building CP-SAT flow model...")
print("=" * 80)

model = cp_model.CpModel()

# Create decision variables for Cargill vessel arcs
cargill_arc_vars = {}
for i, arc in enumerate(cargill_arcs):
    var_name = f"cargill_arc_{arc['vessel_idx']}_{arc['from_node']}_{arc['to_node']}"
    cargill_arc_vars[i] = model.NewBoolVar(var_name)

# Create decision variables for Market vessel assignments
market_arc_vars = {}
for i, arc in enumerate(market_arcs):
    var_name = f"market_arc_{arc['vessel_idx']}_{arc['to_node']}"
    market_arc_vars[i] = model.NewBoolVar(var_name)

print(f"Created {len(cargill_arc_vars)} Cargill vessel arc variables")
print(f"Created {len(market_arc_vars)} Market vessel assignment variables")

# Constraint 1: Each committed cargo must be assigned exactly once
print("\nAdding constraint: Each committed cargo assigned exactly once...")
for c_idx, cargo in cargill_cargoes_processed.iterrows():
    cargo_id = cargo['cargo_id']
    
    # Find all arcs that assign this committed cargo
    cargill_arcs_for_cargo = [
        i for i, arc in enumerate(cargill_arcs)
        if arc['to_node'] == cargo_id and arc['cargo_type'] == 'committed'
    ]
    market_arcs_for_cargo = [
        i for i, arc in enumerate(market_arcs)
        if arc['to_node'] == cargo_id
    ]
    
    # Sum of all assignments to this cargo must equal 1
    terms = []
    for arc_idx in cargill_arcs_for_cargo:
        terms.append(cargill_arc_vars[arc_idx])
    for arc_idx in market_arcs_for_cargo:
        terms.append(market_arc_vars[arc_idx])
    
    if len(terms) > 0:
        model.Add(sum(terms) == 1)
    else:
        print(f"  WARNING: Committed cargo {cargo_id} has no feasible assignments!")

# Constraint 2: Market cargoes are optional (at most once, Cargill vessels only)
print("Adding constraint: Market cargoes optional (at most once)...")
market_cargo_ids = set()
for arc in cargill_arcs:
    if arc['cargo_type'] == 'market':
        market_cargo_ids.add(arc['to_node'])

for cargo_id in market_cargo_ids:
    arcs_for_cargo = [
        i for i, arc in enumerate(cargill_arcs)
        if arc['to_node'] == cargo_id and arc['cargo_type'] == 'market'
    ]
    
    if len(arcs_for_cargo) > 0:
        model.Add(sum(cargill_arc_vars[i] for i in arcs_for_cargo) <= 1)

# Constraint 3: Flow conservation for Cargill vessels
# For each vessel and each cargo node: inflow == outflow
print("Adding constraint: Flow conservation for Cargill vessels...")

# Get all unique cargo nodes (committed + market)
all_cargo_nodes = set()
for arc in cargill_arcs:
    if arc['from_node'] != 'START':
        all_cargo_nodes.add(arc['from_node'])
    all_cargo_nodes.add(arc['to_node'])

# For each Cargill vessel and each cargo node
for v_idx in cargill_vessels_processed.index:
    vessel_name = cargill_vessels_processed.loc[v_idx, 'vessel_name']
    
    for cargo_node in all_cargo_nodes:
        # Inflow: arcs ending at this node for this vessel
        inflow_arcs = [
            i for i, arc in enumerate(cargill_arcs)
            if arc['vessel_idx'] == v_idx and arc['to_node'] == cargo_node
        ]
        
        # Outflow: arcs starting from this node for this vessel
        outflow_arcs = [
            i for i, arc in enumerate(cargill_arcs)
            if arc['vessel_idx'] == v_idx and arc['from_node'] == cargo_node
        ]
        
        # Flow conservation: inflow == outflow
        if len(inflow_arcs) > 0 or len(outflow_arcs) > 0:
            inflow_sum = sum(cargill_arc_vars[i] for i in inflow_arcs)
            outflow_sum = sum(cargill_arc_vars[i] for i in outflow_arcs)
            model.Add(inflow_sum == outflow_sum)

# Constraint 4: Each Cargill vessel can start at most once
# (sum of arcs from START for each vessel <= 1)
print("Adding constraint: Each Cargill vessel starts at most once...")
for v_idx in cargill_vessels_processed.index:
    start_arcs = [
        i for i, arc in enumerate(cargill_arcs)
        if arc['vessel_idx'] == v_idx and arc['from_node'] == 'START'
    ]
    
    if len(start_arcs) > 0:
        model.Add(sum(cargill_arc_vars[i] for i in start_arcs) <= 1)

# Constraint 5: Each Market vessel can be assigned at most once
print("Adding constraint: Each Market vessel assigned at most once...")
for m_idx in market_vessels_processed.index:
    vessel_arcs = [
        i for i, arc in enumerate(market_arcs)
        if arc['vessel_idx'] == m_idx
    ]
    
    if len(vessel_arcs) > 0:
        model.Add(sum(market_arc_vars[i] for i in vessel_arcs) <= 1)

# Objective: Maximize total profit
print("\nSetting objective: Maximize total profit...")
objective_terms = []

# Cargill vessel arcs
for i, arc in enumerate(cargill_arcs):
    profit_scaled = int(round(arc['profit'] * 100))  # Scale to integers
    objective_terms.append(cargill_arc_vars[i] * profit_scaled)

# Market vessel arcs
for i, arc in enumerate(market_arcs):
    profit_scaled = int(round(arc['profit'] * 100))
    objective_terms.append(market_arc_vars[i] * profit_scaled)

model.Maximize(sum(objective_terms))

print("✓ CP-SAT model built successfully")
print(f"  Total variables: {len(cargill_arc_vars) + len(market_arc_vars)}")
print(f"  Objective terms: {len(objective_terms)}")

Building CP-SAT flow model...
Created 20 Cargill vessel arc variables
Created 10 Market vessel assignment variables

Adding constraint: Each committed cargo assigned exactly once...
Adding constraint: Market cargoes optional (at most once)...
Adding constraint: Flow conservation for Cargill vessels...
Adding constraint: Each Cargill vessel starts at most once...
Adding constraint: Each Market vessel assigned at most once...

Setting objective: Maximize total profit...
✓ CP-SAT model built successfully
  Total variables: 30
  Objective terms: 30


In [9]:
# Cell 9: Solve the CP-SAT model

print("Solving CP-SAT model...")
print("=" * 80)

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 300.0  # 5 minutes
solver.parameters.num_search_workers = 4

status = solver.Solve(model)

print(f"\nSolution status: {solver.StatusName(status)}")

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Objective value: ${solver.ObjectiveValue() / 100:.2f}")
    
    # Extract selected arcs
    selected_cargill_arcs = []
    selected_market_arcs = []
    
    for i, arc in enumerate(cargill_arcs):
        if solver.Value(cargill_arc_vars[i]) == 1:
            selected_cargill_arcs.append(arc)
    
    for i, arc in enumerate(market_arcs):
        if solver.Value(market_arc_vars[i]) == 1:
            selected_market_arcs.append(arc)
    
    print(f"\nSelected {len(selected_cargill_arcs)} Cargill vessel arcs")
    print(f"Selected {len(selected_market_arcs)} Market vessel assignments")
    
    # Verify all committed cargoes are assigned
    assigned_committed = set()
    for arc in selected_cargill_arcs:
        if arc['cargo_type'] == 'committed':
            assigned_committed.add(arc['to_node'])
    for arc in selected_market_arcs:
        assigned_committed.add(arc['to_node'])
    
    print(f"\nCommitted cargoes assigned: {len(assigned_committed)}/{len(cargill_cargoes_processed)}")
    for c_idx, cargo in cargill_cargoes_processed.iterrows():
        if cargo['cargo_id'] in assigned_committed:
            print(f"  ✓ {cargo['cargo_id']}")
        else:
            print(f"  ❌ {cargo['cargo_id']} NOT ASSIGNED!")
    
    # Identify market cargoes taken
    assigned_market = set()
    for arc in selected_cargill_arcs:
        if arc['cargo_type'] == 'market':
            assigned_market.add(arc['to_node'])
    
    print(f"\nMarket cargoes taken: {len(assigned_market)}")
    for cargo_id in assigned_market:
        print(f"  ✓ {cargo_id}")
    
else:
    print("No feasible solution found!")
    selected_cargill_arcs = []
    selected_market_arcs = []

Solving CP-SAT model...

Solution status: OPTIMAL
Objective value: $6236785.83

Selected 0 Cargill vessel arcs
Selected 3 Market vessel assignments

Committed cargoes assigned: 3/3
  ✓ CARGILL_1
  ✓ CARGILL_2
  ✓ CARGILL_3

Market cargoes taken: 0


In [10]:
# Cell 10: Reconstruct vessel sequences and generate detailed reports

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Reconstructing vessel sequences...")
    print("=" * 80)
    
    # Reconstruct paths for each Cargill vessel
    vessel_sequences = {}  # vessel_idx -> list of arcs in order
    
    for v_idx in cargill_vessels_processed.index:
        vessel_name = cargill_vessels_processed.loc[v_idx, 'vessel_name']
        vessel_sequences[v_idx] = []
        
        # Find arcs for this vessel
        vessel_arcs = [arc for arc in selected_cargill_arcs if arc['vessel_idx'] == v_idx]
        
        if len(vessel_arcs) == 0:
            continue
        
        # Build sequence by following arcs from START
        # Find starting arc (from START)
        start_arcs = [arc for arc in vessel_arcs if arc['from_node'] == 'START']
        
        if len(start_arcs) > 0:
            # Start with first arc from START
            current_arc = start_arcs[0]
            sequence = [current_arc]
            
            # Follow chain
            while True:
                next_node = current_arc['to_node']
                next_arcs = [arc for arc in vessel_arcs 
                            if arc['from_node'] == next_node and arc not in sequence]
                
                if len(next_arcs) == 0:
                    break
                
                current_arc = next_arcs[0]
                sequence.append(current_arc)
            
            vessel_sequences[v_idx] = sequence
    
    # Build detailed assignment report
    assignments = []
    market_cargo_recommendations = []
    
    # Cargill vessel sequences
    for v_idx, sequence in vessel_sequences.items():
        if len(sequence) == 0:
            continue
        
        vessel_name = cargill_vessels_processed.loc[v_idx, 'vessel_name']
        cumulative_profit = 0
        cumulative_days = 0
        
        for leg_idx, arc in enumerate(sequence):
            leg_data = arc['leg_data']
            cargo_id = arc['to_node']
            cargo_type = arc['cargo_type']
            
            # Get cargo details
            if cargo_type == 'committed':
                cargo = cargill_cargoes_processed[cargill_cargoes_processed['cargo_id'] == cargo_id].iloc[0]
            else:
                cargo = market_cargoes_processed[market_cargoes_processed['cargo_id'] == cargo_id].iloc[0]
            
            cumulative_profit += leg_data['profit']
            cumulative_days += leg_data['total_days']
            
            assignments.append({
                'Vessel_Type': 'Cargill',
                'Vessel_Name': vessel_name,
                'Leg_Number': leg_idx + 1,
                'Cargo_Type': 'Committed' if cargo_type == 'committed' else 'Market',
                'Cargo_ID': cargo_id,
                'Load_Port': cargo['load_port'],
                'Discharge_Port': cargo['discharge_port'],
                'Quantity_MT': cargo['quantity_mt'],
                'Leg_Profit': leg_data['profit'],
                'Cumulative_Profit': cumulative_profit,
                'Leg_Days': leg_data['total_days'],
                'Cumulative_Days': cumulative_days,
                'TCE_Leg': leg_data['profit'] / leg_data['total_days'] if leg_data['total_days'] > 0 else 0,
                'TCE_Cumulative': cumulative_profit / cumulative_days if cumulative_days > 0 else 0,
                'Revenue': leg_data['revenue'],
                'Fuel_Cost': leg_data['fuel_cost'],
                'Fuel_Cost_Ballast': leg_data['fuel_cost_ballast'],
                'Fuel_Cost_Waiting': leg_data['fuel_cost_waiting'],
                'Fuel_Cost_Laden': leg_data['fuel_cost_laden'],
                'Fuel_Cost_Port': leg_data['fuel_cost_port'],
                'Hire_Cost': leg_data['hire_cost'],
                'Port_Costs': leg_data['port_costs'],
                'Commission': leg_data['commission'],
                'Days_Ballast': leg_data['days_ballast'],
                'Days_Waiting': leg_data['days_waiting'],
                'Days_Laden': leg_data['days_laden'],
                'Days_Port': leg_data['days_port'],
                'Ballast_Distance': leg_data['ballast_distance'],
                'Laden_Distance': leg_data['laden_distance'],
                'Start_Time': arc['from_time'],
                'Arrival_Load': leg_data['arrival_at_load'],
                'Actual_Load_Start': leg_data['actual_load_start'],
                'Completion_Time': leg_data['completion_time'],
                'Bunker_Price_Ballast': leg_data['bunker_price_ballast'],
                'Bunker_Price_Waiting': leg_data['bunker_price_waiting'],
                'Bunker_Price_Laden': leg_data['bunker_price_laden'],
                'Bunker_Price_Port': leg_data['bunker_price_port']
            })
            
            # Track market cargoes explicitly
            if cargo_type == 'market':
                market_cargo_recommendations.append({
                    'Vessel_Name': vessel_name,
                    'Leg_Number': leg_idx + 1,
                    'Market_Cargo_ID': cargo_id,
                    'Load_Port': cargo['load_port'],
                    'Discharge_Port': cargo['discharge_port'],
                    'Quantity_MT': cargo['quantity_mt'],
                    'Freight_Rate': cargo['freight_rate_usd_per_mt'],
                    'Leg_Profit': leg_data['profit'],
                    'Leg_TCE': leg_data['profit'] / leg_data['total_days'] if leg_data['total_days'] > 0 else 0,
                    'Incremental_Profit': leg_data['profit'],
                    'Fuel_Cost': leg_data['fuel_cost'],
                    'Hire_Cost': leg_data['hire_cost'],
                    'Total_Days': leg_data['total_days'],
                    'Position_In_Sequence': f"After leg {leg_idx}" if leg_idx > 0 else "First leg",
                    'Previous_Cargo': sequence[leg_idx-1]['to_node'] if leg_idx > 0 else 'START'
                })
    
    # Market vessel assignments
    for arc in selected_market_arcs:
        vessel_name = arc['vessel_name']
        cargo_id = arc['to_node']
        cargo = cargill_cargoes_processed[cargill_cargoes_processed['cargo_id'] == cargo_id].iloc[0]
        leg_data = arc['leg_data']
        
        assignments.append({
            'Vessel_Type': 'Market',
            'Vessel_Name': vessel_name,
            'Leg_Number': 1,
            'Cargo_Type': 'Committed',
            'Cargo_ID': cargo_id,
            'Load_Port': cargo['load_port'],
            'Discharge_Port': cargo['discharge_port'],
            'Quantity_MT': cargo['quantity_mt'],
            'Leg_Profit': leg_data['profit'],
            'Cumulative_Profit': leg_data['profit'],
            'Leg_Days': leg_data['total_days'],
            'Cumulative_Days': leg_data['total_days'],
            'TCE_Leg': leg_data['profit'] / leg_data['total_days'] if leg_data['total_days'] > 0 else 0,
            'TCE_Cumulative': leg_data['profit'] / leg_data['total_days'] if leg_data['total_days'] > 0 else 0,
            'Revenue': leg_data['revenue'],
            'Fuel_Cost': leg_data['fuel_cost'],
            'Fuel_Cost_Ballast': leg_data['fuel_cost_ballast'],
            'Fuel_Cost_Waiting': leg_data['fuel_cost_waiting'],
            'Fuel_Cost_Laden': leg_data['fuel_cost_laden'],
            'Fuel_Cost_Port': leg_data['fuel_cost_port'],
            'Hire_Cost': leg_data['hire_cost'],
            'Port_Costs': leg_data['port_costs'],
            'Commission': leg_data['commission'],
            'Days_Ballast': leg_data['days_ballast'],
            'Days_Waiting': leg_data['days_waiting'],
            'Days_Laden': leg_data['days_laden'],
            'Days_Port': leg_data['days_port'],
            'Ballast_Distance': leg_data['ballast_distance'],
            'Laden_Distance': leg_data['laden_distance'],
            'Start_Time': arc['from_time'],
            'Arrival_Load': leg_data['arrival_at_load'],
            'Actual_Load_Start': leg_data['actual_load_start'],
            'Completion_Time': leg_data['completion_time'],
            'Bunker_Price_Ballast': leg_data['bunker_price_ballast'],
            'Bunker_Price_Waiting': leg_data['bunker_price_waiting'],
            'Bunker_Price_Laden': leg_data['bunker_price_laden'],
            'Bunker_Price_Port': leg_data['bunker_price_port']
        })
    
    assignments_df = pd.DataFrame(assignments)
    market_recos_df = pd.DataFrame(market_cargo_recommendations)
    
    print(f"\n✓ Generated detailed reports")
    print(f"  Total assignments: {len(assignments_df)}")
    print(f"  Market cargo recommendations: {len(market_recos_df)}")
    
else:
    assignments_df = pd.DataFrame()
    market_recos_df = pd.DataFrame()

Reconstructing vessel sequences...

✓ Generated detailed reports
  Total assignments: 3
  Market cargo recommendations: 0


In [11]:
# Cell 11: Display results - Market Cargo Recommendations (EXPLICIT OUTPUT)

if len(market_recos_df) > 0:
    print("=" * 80)
    print("MARKET CARGO RECOMMENDATIONS")
    print("=" * 80)
    print("\nThe following market cargoes should be taken to maximize profit:\n")
    
    for idx, row in market_recos_df.iterrows():
        print(f"{idx+1}. {row['Market_Cargo_ID']}")
        print(f"   Vessel: {row['Vessel_Name']}")
        print(f"   Position: {row['Position_In_Sequence']} (Leg {row['Leg_Number']})")
        if row['Previous_Cargo'] != 'START':
            print(f"   After completing: {row['Previous_Cargo']}")
        print(f"   Route: {row['Load_Port']} → {row['Discharge_Port']}")
        print(f"   Quantity: {row['Quantity_MT']:,.0f} MT")
        print(f"   Freight Rate: ${row['Freight_Rate']:.2f}/MT")
        print(f"   Incremental Profit: ${row['Incremental_Profit']:,.2f}")
        print(f"   Leg TCE: ${row['Leg_TCE']:,.2f}/day")
        print(f"   Fuel Cost: ${row['Fuel_Cost']:,.2f}")
        print(f"   Hire Cost: ${row['Hire_Cost']:,.2f}")
        print(f"   Total Days: {row['Total_Days']:.1f} days")
        print()
    
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Total market cargoes recommended: {len(market_recos_df)}")
    print(f"Total incremental profit from market cargoes: ${market_recos_df['Incremental_Profit'].sum():,.2f}")
    print(f"Average TCE for market cargoes: ${market_recos_df['Leg_TCE'].mean():,.2f}/day")
    
else:
    print("=" * 80)
    print("MARKET CARGO RECOMMENDATIONS")
    print("=" * 80)
    print("\nNo market cargoes recommended in optimal solution.")
    print("All vessels are optimally assigned to committed cargoes only.")

MARKET CARGO RECOMMENDATIONS

No market cargoes recommended in optimal solution.
All vessels are optimally assigned to committed cargoes only.


In [12]:
# Cell 12: Display results - Complete Assignment Summary

if len(assignments_df) > 0:
    print("=" * 80)
    print("COMPLETE ASSIGNMENT SUMMARY")
    print("=" * 80)
    
    # Group by vessel
    for vessel_name in assignments_df['Vessel_Name'].unique():
        vessel_assignments = assignments_df[assignments_df['Vessel_Name'] == vessel_name].sort_values('Leg_Number')
        vessel_type = vessel_assignments.iloc[0]['Vessel_Type']
        
        print(f"\n{'='*80}")
        print(f"{vessel_type} Vessel: {vessel_name}")
        print(f"{'='*80}")
        
        total_profit = vessel_assignments['Leg_Profit'].sum()
        total_days = vessel_assignments['Leg_Days'].sum()
        total_tce = total_profit / total_days if total_days > 0 else 0
        
        print(f"\nTotal Voyage Profit: ${total_profit:,.2f}")
        print(f"Total Voyage Days: {total_days:.1f} days")
        print(f"Overall TCE: ${total_tce:,.2f}/day")
        
        print(f"\nLeg-by-Leg Breakdown:")
        print("-" * 80)
        print(f"{'Leg':<5} {'Cargo':<15} {'Type':<10} {'Load Port':<20} {'Discharge Port':<20} {'Profit':>12} {'TCE':>12} {'Days':>8}")
        print("-" * 80)
        
        for _, leg in vessel_assignments.iterrows():
            cargo_type_short = leg['Cargo_Type'][:8]
            print(f"{leg['Leg_Number']:<5} {leg['Cargo_ID']:<15} {cargo_type_short:<10} {leg['Load_Port']:<20} {leg['Discharge_Port']:<20} ${leg['Leg_Profit']:>11,.0f} ${leg['TCE_Leg']:>11,.0f} {leg['Leg_Days']:>7.1f}")
        
        # Fuel breakdown
        print(f"\nFuel Cost Breakdown:")
        print(f"  Ballast: ${vessel_assignments['Fuel_Cost_Ballast'].sum():,.2f}")
        print(f"  Waiting: ${vessel_assignments['Fuel_Cost_Waiting'].sum():,.2f}")
        print(f"  Laden: ${vessel_assignments['Fuel_Cost_Laden'].sum():,.2f}")
        print(f"  Port: ${vessel_assignments['Fuel_Cost_Port'].sum():,.2f}")
        print(f"  Total Fuel: ${vessel_assignments['Fuel_Cost'].sum():,.2f}")
    
    # Overall summary
    print(f"\n{'='*80}")
    print("OVERALL SUMMARY")
    print(f"{'='*80}")
    total_profit_all = assignments_df['Leg_Profit'].sum()
    total_days_all = assignments_df.groupby('Vessel_Name')['Leg_Days'].sum().sum()
    
    cargill_profit = assignments_df[assignments_df['Vessel_Type'] == 'Cargill']['Leg_Profit'].sum()
    market_profit = assignments_df[assignments_df['Vessel_Type'] == 'Market']['Leg_Profit'].sum()
    
    committed_profit = assignments_df[assignments_df['Cargo_Type'] == 'Committed']['Leg_Profit'].sum()
    market_cargo_profit = assignments_df[assignments_df['Cargo_Type'] == 'Market']['Leg_Profit'].sum()
    
    print(f"\nTotal Profit: ${total_profit_all:,.2f}")
    print(f"  From Cargill vessels: ${cargill_profit:,.2f}")
    print(f"  From Market vessels: ${market_profit:,.2f}")
    print(f"\nProfit Breakdown by Cargo Type:")
    print(f"  Committed cargoes: ${committed_profit:,.2f}")
    print(f"  Market cargoes: ${market_cargo_profit:,.2f}")
    print(f"\nTotal Voyage Days: {total_days_all:.1f} days")
    print(f"Overall TCE: ${total_profit_all/total_days_all:,.2f}/day" if total_days_all > 0 else "N/A")
    
else:
    print("No assignments to display.")

COMPLETE ASSIGNMENT SUMMARY

Market Vessel: ATLANTIC FORTUNE

Total Voyage Profit: $564,564.49
Total Voyage Days: 23.1 days
Overall TCE: $24,399.38/day

Leg-by-Leg Breakdown:
--------------------------------------------------------------------------------
Leg   Cargo           Type       Load Port            Discharge Port             Profit          TCE     Days
--------------------------------------------------------------------------------
1     CARGILL_2       Committe   PORT HEDLAND         LIANYUNGANG          $    564,564 $     24,399    23.1

Fuel Cost Breakdown:
  Ballast: $186,335.87
  Waiting: $0.00
  Laden: $252,896.09
  Port: $2,203.55
  Total Fuel: $441,435.51

Market Vessel: CORAL EMPEROR

Total Voyage Profit: $2,624,078.87
Total Voyage Days: 68.3 days
Overall TCE: $38,419.67/day

Leg-by-Leg Breakdown:
--------------------------------------------------------------------------------
Leg   Cargo           Type       Load Port            Discharge Port             Profit   

In [ ]:
%pip install matplotlib

In [15]:
# Cell 13: Visualize multi-leg routes on map

if len(assignments_df) > 0:
    import folium
    from folium import plugins
    
    print("Generating route visualization map...")
    
    # Create base map centered on average of all ports
    all_lats = []
    all_lons = []
    for _, port in port_locations.iterrows():
        all_lats.append(port['latitude'])
        all_lons.append(port['longitude'])
    
    center_lat = sum(all_lats) / len(all_lats)
    center_lon = sum(all_lons) / len(all_lons)
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=3)
    
    # Color palette for vessels
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    vessel_names = assignments_df['Vessel_Name'].unique()
    colors = cm.tab20(range(len(vessel_names)))
    
    vessel_colors = {}
    for i, vessel_name in enumerate(vessel_names):
        vessel_colors[vessel_name] = mcolors.rgb2hex(colors[i])
    
    # Plot routes for each vessel
    for vessel_name in vessel_names:
        vessel_assignments = assignments_df[assignments_df['Vessel_Name'] == vessel_name].sort_values('Leg_Number')
        vessel_type = vessel_assignments.iloc[0]['Vessel_Type']
        color = vessel_colors[vessel_name]
        
        # Get vessel start position
        if vessel_type == 'Cargill':
            vessel_row = cargill_vessels_processed[cargill_vessels_processed['vessel_name'] == vessel_name].iloc[0]
        else:
            vessel_row = market_vessels_processed[market_vessels_processed['vessel_name'] == vessel_name].iloc[0]
        
        start_port = vessel_row['current_position_port']
        start_port_coords = port_locations[port_locations['port_name'] == start_port]
        
        if len(start_port_coords) > 0:
            start_lat = start_port_coords.iloc[0]['latitude']
            start_lon = start_port_coords.iloc[0]['longitude']
            
            # Add start marker
            folium.Marker(
                [start_lat, start_lon],
                popup=f"{vessel_name} START",
                icon=folium.Icon(color='green', icon='ship', prefix='fa')
            ).add_to(m)
        
        # Plot each leg
        route_coords = []
        if len(start_port_coords) > 0:
            route_coords.append([start_lat, start_lon])
        
        for _, leg in vessel_assignments.iterrows():
            load_port = leg['Load_Port']
            discharge_port = leg['Discharge_Port']
            cargo_id = leg['Cargo_ID']
            cargo_type = leg['Cargo_Type']
            
            # Get coordinates
            load_coords = port_locations[port_locations['port_name'] == load_port]
            disc_coords = port_locations[port_locations['port_name'] == discharge_port]
            
            if len(load_coords) > 0 and len(disc_coords) > 0:
                load_lat = load_coords.iloc[0]['latitude']
                load_lon = load_coords.iloc[0]['longitude']
                disc_lat = disc_coords.iloc[0]['latitude']
                disc_lon = disc_coords.iloc[0]['longitude']
                
                # Add load port marker
                marker_color = 'blue' if cargo_type == 'Committed' else 'orange'
                folium.Marker(
                    [load_lat, load_lon],
                    popup=f"{cargo_id}<br>{cargo_type}<br>Load: {load_port}",
                    icon=folium.Icon(color=marker_color, icon='upload', prefix='fa')
                ).add_to(m)
                
                # Add discharge port marker
                folium.Marker(
                    [disc_lat, disc_lon],
                    popup=f"{cargo_id}<br>{cargo_type}<br>Discharge: {discharge_port}",
                    icon=folium.Icon(color='red', icon='download', prefix='fa')
                ).add_to(m)
                
                # Add route line
                route_coords.append([load_lat, load_lon])
                route_coords.append([disc_lat, disc_lon])
                
                # Draw line for this leg
                folium.PolyLine(
                    [[load_lat, load_lon], [disc_lat, disc_lon]],
                    color=color,
                    weight=3,
                    opacity=0.7,
                    popup=f"{vessel_name} Leg {leg['Leg_Number']}: {cargo_id}"
                ).add_to(m)
        
        # Add vessel route summary
        if len(route_coords) > 1:
            folium.PolyLine(
                route_coords,
                color=color,
                weight=2,
                opacity=0.5,
                dashArray='5, 5',
                popup=f"{vessel_name} Route"
            ).add_to(m)
    
    # Add legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 200px; height: auto; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:14px; padding: 10px">
    <h4>Legend</h4>
    <p><i class="fa fa-ship" style="color:green"></i> Vessel Start</p>
    <p><i class="fa fa-upload" style="color:blue"></i> Committed Load</p>
    <p><i class="fa fa-upload" style="color:orange"></i> Market Load</p>
    <p><i class="fa fa-download" style="color:red"></i> Discharge</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Save map
    map_filename = 'vessel_routes_multileg.html'
    m.save(map_filename)
    print(f"✓ Map saved to: {map_filename}")
    print(f"  Open {map_filename} in a web browser to view the routes")
    
    # Display in notebook (if running in Jupyter)
    try:
        from IPython.display import HTML
        display(m)
    except:
        pass
    
else:
    print("No routes to visualize.")

Generating route visualization map...
✓ Map saved to: vessel_routes_multileg.html
  Open vessel_routes_multileg.html in a web browser to view the routes


In [ ]:
# Cell 14: Save results to CSV files

if len(assignments_df) > 0:
    print("Saving results to CSV files...")
    
    # Save complete assignments
    assignments_df.to_csv('multileg_assignments.csv', index=False)
    print(f"✓ Saved assignments to: multileg_assignments.csv")
    
    # Save market cargo recommendations
    if len(market_recos_df) > 0:
        market_recos_df.to_csv('market_cargo_recommendations.csv', index=False)
        print(f"✓ Saved market cargo recommendations to: market_cargo_recommendations.csv")
    
    # Create summary by vessel
    vessel_summary = []
    for vessel_name in assignments_df['Vessel_Name'].unique():
        vessel_assignments = assignments_df[assignments_df['Vessel_Name'] == vessel_name]
        vessel_type = vessel_assignments.iloc[0]['Vessel_Type']
        
        total_profit = vessel_assignments['Leg_Profit'].sum()
        total_days = vessel_assignments['Leg_Days'].sum()
        total_tce = total_profit / total_days if total_days > 0 else 0
        
        committed_count = len(vessel_assignments[vessel_assignments['Cargo_Type'] == 'Committed'])
        market_count = len(vessel_assignments[vessel_assignments['Cargo_Type'] == 'Market'])
        
        vessel_summary.append({
            'Vessel_Type': vessel_type,
            'Vessel_Name': vessel_name,
            'Total_Legs': len(vessel_assignments),
            'Committed_Cargoes': committed_count,
            'Market_Cargoes': market_count,
            'Total_Profit': total_profit,
            'Total_Days': total_days,
            'Overall_TCE': total_tce,
            'Total_Fuel_Cost': vessel_assignments['Fuel_Cost'].sum(),
            'Total_Hire_Cost': vessel_assignments['Hire_Cost'].sum(),
            'Total_Port_Costs': vessel_assignments['Port_Costs'].sum()
        })
    
    summary_df = pd.DataFrame(vessel_summary)
    summary_df.to_csv('multileg_vessel_summary.csv', index=False)
    print(f"✓ Saved vessel summary to: multileg_vessel_summary.csv")
    
    print("\n✓ All results saved successfully!")
    
else:
    print("No results to save.")